# Neural Networks in PyTorch - Regularization Techniques

In [ ]:
import numpy as np
from matplotlib import pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms

# Set random seed for reproducibility
torch.manual_seed(302)
np.random.seed(302)

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_dataset = datasets.MNIST('./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST('./data', train=False, transform=transform)

train_dataset, val_dataset = torch.utils.data.random_split(train_dataset, [50000, 10000])

print(train_dataset); print(test_dataset)

# Creating data loaders
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=64, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=64, shuffle=True)

In [ ]:
class NeuralNet(nn.Module):
    def __init__(self):
      super(NeuralNet, self).__init__()
      self.fc1 = nn.Linear(in_features=28*28, out_features=128)
      self.fc2 = nn.Linear(in_features=128, out_features=128)
      self.fc3 = nn.Linear(in_features=128, out_features=10)

    def forward(self, x):
      # Flatten the data (B, 1, 28, 28) => (B, 784), where B is the batch size
      x = torch.flatten(x, start_dim=1)

      # Pass data through 1st fully connected layer
      x = self.fc1(x)
      # Apply ReLU non-linearity
      x = F.relu(x)

      # Pass data through 2nd fully connected layer
      x = self.fc2(x)
      # Apply ReLU non-linearity
      x = F.relu(x)

      # Pass data through 3rd fully connected layer
      x = self.fc3(x)

      # The output values in x are called *logits*
      return x

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

model = NeuralNet().to(device)
print(model)

# Parameters of the model
for n, p in model.named_parameters():
    print(n, p.shape)

## (1) Parameter norm penalties: Example with L1 Regularization

In [ ]:
def l1_regularization_loss(params, lamda=0.001):
    weights = [p for p in params if 'weight' in n]
    return lamda * sum(abs(p).sum() for p in weights)


def train_model_with_parameter_reg(model, optimizer, train_loader, val_loader, device, num_epochs):
    loss_fn = nn.CrossEntropyLoss()

    train_losses = []
    val_losses = []
    val_accuracies = []
    for epoch in range(num_epochs):
        print('-'*20, f'Epoch {epoch}', '-'*20)

        # Train one epoch
        model.train()
        for batch_idx, (data, target) in enumerate(train_loader):
            data, target = data.to(device), target.to(device)

            optimizer.zero_grad()
            outputs = model(data)
            loss = loss_fn(outputs, target) + l1_regularization_loss(model.parameters(), lamda=0.001)
            loss.backward()
            optimizer.step()

            train_losses.append(loss.item())

        print(f'Train Epoch {epoch} | Average Training Loss {np.mean(train_losses[-len(train_loader):])}')

        # Evaluate on validation set at the end of the epoch
        model.eval()
        val_loss = 0
        correct = 0
        with torch.no_grad():
            for data, target in val_loader:
                data, target = data.to(device), target.to(device)
                outputs = model(data)
                val_loss += loss_fn(outputs, target).item()
                probs = F.softmax(outputs, dim=1)
                pred = torch.argmax(probs, dim=1)  # get the index of the max probability as the predicted output
                correct += (pred == target).sum().item()

        val_loss = val_loss / len(val_loader)
        avg_correct = correct / len(val_loader.dataset)
        val_losses.append(val_loss)
        val_accuracies.append(avg_correct)
        print(f'Validation set: Average loss: {val_loss:.4f}, Accuracy: {correct}/{len(val_loader.dataset)} ({100. * avg_correct:.0f}%)\n')

    return train_losses, val_losses, val_accuracies

In [ ]:
model = NeuralNet().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.002)

train_losses, val_losses, val_accuracies = train_model_with_parameter_reg(model, optimizer, train_loader, val_loader, device, num_epochs=50)

## (2) Early Stopping

In [ ]:
class EarlyStopping:
    def __init__(self, patience=10):
        self.patience = patience
        self.counter = 0
        self.best_loss = np.inf

    def should_stop(self, val_loss):
        if val_loss < self.best_loss:
            self.best_loss = val_loss
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                print(f"Early stopping triggered. No improvement for {self.patience} epochs.")
                return True
        return False

In [ ]:
def train_model_with_early_stopping(model, optimizer, train_loader, val_loader, device, num_epochs, early_stopping):
    loss_fn = nn.CrossEntropyLoss()

    train_losses = []
    val_losses = []
    val_accuracies = []
    for epoch in range(num_epochs):
        print('-'*20, f'Epoch {epoch}', '-'*20)

        # Train one epoch
        model.train()
        for batch_idx, (data, target) in enumerate(train_loader):
            data, target = data.to(device), target.to(device)

            optimizer.zero_grad()
            outputs = model(data)
            loss = loss_fn(outputs, target)
            loss.backward()
            optimizer.step()

            train_losses.append(loss.item())

        print(f'Train Epoch {epoch} | Average Training Loss {np.mean(train_losses[-len(train_loader):])}')

        # Evaluate on validation set at the end of the epoch
        model.eval()
        val_loss = 0
        correct = 0
        with torch.no_grad():
            for data, target in val_loader:
                data, target = data.to(device), target.to(device)
                outputs = model(data)
                val_loss += loss_fn(outputs, target).item()
                probs = F.softmax(outputs, dim=1)
                pred = torch.argmax(probs, dim=1)  # get the index of the max probability as the predicted output
                correct += (pred == target).sum().item()

        val_loss = val_loss / len(val_loader)
        avg_correct = correct / len(val_loader.dataset)
        val_losses.append(val_loss)
        val_accuracies.append(avg_correct)
        print(f'Validation set: Average loss: {val_loss:.4f}, Accuracy: {correct}/{len(val_loader.dataset)} ({100. * avg_correct:.0f}%)\n')

        # Early Stopping
        if early_stopping and early_stopping.should_stop(val_loss):
            print(f"Stopping early at epoch {epoch+1}")
            break

    return train_losses, val_losses, val_accuracies

In [ ]:
model = NeuralNet().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.002)
train_losses, train_accuracies, val_losses, val_accuracies = train_model_with_early_stopping(model, optimizer, train_loader, val_loader, device,
                                                                                             num_epochs=50, early_stopping=EarlyStopping(patience=5))

## (3) Dropout

In [ ]:
class DropoutNet(nn.Module):
    def __init__(self, p=0.25):
        super(DropoutNet, self).__init__()
        self.fcs = nn.Sequential(
            nn.Linear(in_features=28*28, out_features=64),
            nn.ReLU(),
            nn.Dropout(p=p),
            nn.Linear(in_features=64, out_features=128),
            nn.ReLU(),
            nn.Dropout(p=p),
            nn.Linear(in_features=128, out_features=128),
            nn.ReLU(),
            nn.Dropout(p=p),
            nn.Linear(in_features=128, out_features=128),
            nn.ReLU(),
            nn.Dropout(p=p),
            nn.Linear(in_features=128, out_features=64),
            nn.ReLU(),
            nn.Dropout(p=p),
            nn.Linear(in_features=64, out_features=32),
            nn.ReLU(),
            nn.Dropout(p=p),
            nn.Linear(in_features=32, out_features=10)
        )

    def forward(self, x):
        x = torch.flatten(x, start_dim=1)
        x = self.fcs(x)
        return outputs

In [ ]:
model = DropoutNet(p=0.15).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.002)
cross_entropy_loss = nn.CrossEntropyLoss()

train_losses, val_losses, val_accuracies = train_model_with_early_stopping(model, optimizer, train_loader, val_loader, device,
                                                                           num_epochs=50, early_stopping=EarlyStopping(patience=5))


## (4) Data Augmentation via torchvision.transforms

https://docs.pytorch.org/vision/0.15/transforms.html

https://docs.pytorch.org/vision/0.15/auto_examples/plot_transforms.html#illustration-of-transforms


In [ ]:
import numpy as np
from matplotlib import pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms

# Set random seed for reproducibility
torch.manual_seed(302)
np.random.seed(302)

transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.RandomRotation(15),               # small random rotations
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),  # mean for each channel
                         (0.2023, 0.1994, 0.2010))  # std for each channel
])

train_dataset = datasets.CIFAR10('./data', train=True, download=True, transform=transform)
test_dataset = datasets.CIFAR10('./data', train=False, transform=transform)
